## Feature Engineering
Building powerful signals from raw transaction data.

In [5]:
import pandas as pd
import numpy as np

In [6]:
train = pd.read_csv("../data/raw/fraudTrain.csv")
test = pd.read_csv("../data/raw/fraudTest.csv")

In [7]:
for df in [train, test]:
    df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

print('Train:', train.shape, '| Test:', test.shape)

Train: (1296675, 22) | Test: (555719, 22)


## DateTime Features

In [8]:
for df in [train, test]:
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['dob']                   = pd.to_datetime(df['dob'])

    df['hour']        = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month']       = df['trans_date_trans_time'].dt.month
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
    df['is_night']    = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    # Age at time of transaction
    df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

print('Datetime features done.')
train[['hour','day_of_week','is_weekend','is_night','age']].head()

Datetime features done.


,hour,day_of_week,is_weekend,is_night,age
0,0,1,0,1,30
1,0,1,0,1,40
2,0,1,0,1,56
3,0,1,0,1,52
4,0,1,0,1,32


## Geographic Distance (Haversine)

In [9]:
def haversine(lat1, lon1, lat2, lon2):
    """Haversine formula — great-circle distance in km."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

In [10]:
for df in [train, test]:
    df['geo_distance'] = haversine(
        df['lat'], df['long'],
        df['merch_lat'], df['merch_long']
    )

print('Geo distance :- Train stats:')
print(train['geo_distance'].describe())

Geo distance :- Train stats:
count    1.296675e+06
mean     7.611465e+01
std      2.911693e+01
min      2.225452e-02
25%      5.533491e+01
50%      7.823175e+01
75%      9.850327e+01
max      1.521172e+02
Name: geo_distance, dtype: float64


## Amount Z-Score (per Cardholder)

In [11]:
card_stats = (train.groupby('cc_num')['amt'].agg(['mean', 'std']).rename(columns={'mean': 'card_mean_amt', 'std': 'card_std_amt'}).reset_index())

In [12]:
for df in [train, test]:
    df = df.merge(card_stats, on='cc_num', how='left')
    df['card_std_amt'].fillna(1, inplace=True)   # cards with single txn
    df['amt_zscore'] = (df['amt'] - df['card_mean_amt']) / df['card_std_amt']
    df['amt_zscore'].fillna(0, inplace=True)

# Need to re-merge properly (merge returns a new df)
train = train.merge(card_stats, on='cc_num', how='left')
train['card_std_amt'].fillna(1, inplace=True)
train['amt_zscore'] = (train['amt'] - train['card_mean_amt']) / train['card_std_amt']
train['amt_zscore'].fillna(0, inplace=True)

test = test.merge(card_stats, on='cc_num', how='left')
test['card_std_amt'].fillna(1, inplace=True)
test['amt_zscore'] = (test['amt'] - test['card_mean_amt']) / test['card_std_amt']
test['amt_zscore'].fillna(0, inplace=True)

C:\Users\OJ\AppData\Local\Temp\ipykernel_304\1000573097.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['card_std_amt'].fillna(1, inplace=True)   # cards with single txn
C:\Users\OJ\AppData\Local\Temp\ipykernel_304\1000573097.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained 

0        -0.571146
1        -0.187884
2        -0.389891
3         0.012580
4        -0.585185
            ...   
555714   -0.172453
555715    0.316596
555716   -0.001429
555717   -0.579047
555718   -0.146791
Name: amt_zscore, Length: 555719, dtype: float64

In [14]:
print('Amount z-score sample:')
train[['cc_num','amt','card_mean_amt','amt_zscore']].head()

Amount z-score sample:


,cc_num,amt,card_mean_amt,amt_zscore
0,2703186189652095,4.97,87.393215,-0.651072
1,630423337322,107.23,53.949320,0.450243
2,38859492057661,220.11,65.870040,1.518323
3,3534093764340240,45.00,72.776673,-0.186931
4,375534208663984,41.96,95.178091,-0.597057


## Encoding Categorical Features

In [15]:
from sklearn.preprocessing import LabelEncoder

In [23]:
le_cat    = LabelEncoder()
le_gender = LabelEncoder()

train['category_enc'] = le_cat.fit_transform(train['category'])
test['category_enc']  = le_cat.transform(test['category'].map(
    lambda x: x if x in le_cat.classes_ else le_cat.classes_[0]))

# Category fraud rate encoding (target encoding from train only)
cat_fraud_rate = train.groupby('category')['is_fraud'].mean().to_dict()
train['category_fraud_rate'] = train['category'].map(cat_fraud_rate)
test['category_fraud_rate']  = test['category'].map(cat_fraud_rate).fillna(0)

print('Categories encoded:', le_cat.classes_)

Categories encoded: ['entertainment' 'food_dining' 'gas_transport' 'grocery_net' 'grocery_pos'
 'health_fitness' 'home' 'kids_pets' 'misc_net' 'misc_pos' 'personal_care'
 'shopping_net' 'shopping_pos' 'travel']


## Final Feature Selection

In [20]:
FEATURES = [
    # Raw numeric
    'amt',
    # Time features
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_night',
    # Geographic
    'geo_distance', 'lat', 'long', 'merch_lat', 'merch_long',
    # Amount behavior
    'amt_zscore', 'card_mean_amt',
    # Merchant
    'category_enc', 'category_fraud_rate',
]

In [21]:
TARGET = 'is_fraud'
X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)
print('\nFeatures used:')
for f in FEATURES:
    print(' -', f)

X_train shape: (1296675, 15)
X_test  shape: (555719, 15)

Features used:
 - amt
 - hour
 - day_of_week
 - month
 - is_weekend
 - is_night
 - geo_distance
 - lat
 - long
 - merch_lat
 - merch_long
 - amt_zscore
 - card_mean_amt
 - category_enc
 - category_fraud_rate


## Save Processed Data

In [27]:
# Fill any remaining NaNs with 0
X_train = X_train.fillna(0)
X_test  = X_test.fillna(0)

print("NaNs remaining in X_train:", X_train.isnull().sum().sum())
print("NaNs remaining in X_test:",  X_test.isnull().sum().sum())

NaNs remaining in X_train: 0
NaNs remaining in X_test: 0


In [28]:
import os
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

In [29]:
# Save feature list for use in training
import json
with open('../data/processed/features.json', 'w') as f:
    json.dump(FEATURES, f)

In [30]:
print('Saved processed data to data/processed/')
print('→ Proceed to 03_model_training.ipynb')

Saved processed data to data/processed/
→ Proceed to 03_model_training.ipynb
